# 🧠 Deepfake Audio Detection — Baseline CNN
**Notebook:** `02_baseline_model.ipynb`  
**Team:** Kazim Mammadli · Vidadi Javadov · Zamin Sultanli  
**Purpose:** End-to-end sanity check — train a lightweight 3-block CNN on Mel spectrograms to confirm the pipeline feeds a model correctly before moving to pretrained networks.

### What this notebook does
1. Environment & imports  
2. Load TF datasets  
3. Apply preprocessing adapter (`custom` mode → channels-last)  
4. Build 3-block CNN  
5. Train with EarlyStopping + ModelCheckpoint  
6. Plot accuracy & loss curves  
7. Evaluate on test set: Accuracy, F1, AUC-ROC, Confusion Matrix

---
## 1. Environment & Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

# Pipeline
from src.features.mel_spectrogram import extract, set_preset, _cfg
from src.data.dataset import FoRDataset, build_tf_datasets

# Shared utilities (no inline duplication)
from src.training.preprocessing import preprocess_for_model
from src.evaluation.visualize import plot_history, plot_confusion_matrix
from src.evaluation.metrics import evaluate_model

set_preset('for-2sec')

print(f'Python     : {sys.version.split()[0]}')
print(f'TensorFlow : {tf.__version__}')
print(f'Mel config : {_cfg}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')
print()
print('✅ All imports successful')

---
## 2. Load TF Datasets

In [ ]:
DATASET_DIR = '../data/raw/for-2seconds'
BATCH_SIZE  = 32

print('Building TF datasets...')
train_ds, val_ds, test_ds = build_tf_datasets(
    DATASET_DIR, preset='for-2sec', batch_size=BATCH_SIZE
)

for x, y in train_ds.take(1):
    print(f'\nRaw batch — x: {x.shape}, dtype={x.dtype} | y: {y.shape}, dtype={y.dtype}')
    print(f'x range   : [{x.numpy().min():.4f}, {x.numpy().max():.4f}]')
    print(f'y sample  : {y.numpy()[:8]}')
print('\n✅ Datasets loaded')

---
## 3. Apply Preprocessing Adapter

`preprocess_for_model` is defined once in `src/training/preprocessing.py` and imported here.  
For this notebook we use **`'custom'` mode**: transpose only → `(batch, 80, target_frames, 1)`.

In [ ]:
# preprocess_for_model is imported from src.training.preprocessing
train_custom = train_ds.map(
    lambda x, y: preprocess_for_model(x, y, 'custom'),
    num_parallel_calls=tf.data.AUTOTUNE
)
val_custom = val_ds.map(
    lambda x, y: preprocess_for_model(x, y, 'custom'),
    num_parallel_calls=tf.data.AUTOTUNE
)
test_custom = test_ds.map(
    lambda x, y: preprocess_for_model(x, y, 'custom'),
    num_parallel_calls=tf.data.AUTOTUNE
)

for x, y in train_custom.take(1):
    print(f'Custom batch — x: {x.shape}   (expected: ({BATCH_SIZE}, {_cfg["n_mels"]}, {_cfg["target_frames"]}, 1))')
    print(f'              y: {y.shape}')
print('\n✅ Preprocessing adapter applied (custom mode)')

---
## 4. Build 3-Block CNN

In [ ]:
def build_baseline_cnn(input_shape):
    """3-block CNN for Mel spectrogram binary classification."""
    inputs = keras.Input(shape=input_shape, name='mel_input')

    # Block 1
    x = keras.layers.Conv2D(32, 3, padding='same', activation='relu', name='conv1')(inputs)
    x = keras.layers.BatchNormalization(name='bn1')(x)
    x = keras.layers.MaxPooling2D(name='pool1')(x)

    # Block 2
    x = keras.layers.Conv2D(64, 3, padding='same', activation='relu', name='conv2')(x)
    x = keras.layers.BatchNormalization(name='bn2')(x)
    x = keras.layers.MaxPooling2D(name='pool2')(x)

    # Block 3
    x = keras.layers.Conv2D(128, 3, padding='same', activation='relu', name='conv3')(x)
    x = keras.layers.BatchNormalization(name='bn3')(x)
    x = keras.layers.MaxPooling2D(name='pool3')(x)

    # Classification head
    x = keras.layers.GlobalAveragePooling2D(name='gap')(x)
    x = keras.layers.Dropout(0.3, name='dropout')(x)
    outputs = keras.layers.Dense(1, activation='sigmoid', name='output')(x)

    return keras.Model(inputs, outputs, name='baseline_cnn')


# Input shape: (n_mels, target_frames, 1) — channels-last
input_shape = (_cfg['n_mels'], _cfg['target_frames'], 1)
model = build_baseline_cnn(input_shape)

model.compile(
    optimizer=keras.optimizers.Adam(),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()
print(f'\nInput shape : {input_shape}')
print(f'Total params: {model.count_params():,}')

---
## 5. Train

In [ ]:
os.makedirs('../models', exist_ok=True)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath='../models/baseline_cnn.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
]

print('Starting training...')
history = model.fit(
    train_custom,
    validation_data=val_custom,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)
print('\n✅ Training complete')

---
## 6. Training Curves

`plot_history` is defined once in `src/evaluation/visualize.py` and imported here.

In [ ]:
# plot_history is imported from src.evaluation.visualize
os.makedirs('../outputs', exist_ok=True)
plot_history(
    history,
    title='Baseline CNN — Training History',
    save_path='../outputs/baseline_cnn_curves.png'
)

---
## 7. Evaluate on Test Set

`evaluate_model` is defined once in `src/evaluation/metrics.py` and imported here.  
It collects predictions, prints Accuracy / F1 / AUC-ROC, and saves the confusion matrix.

In [ ]:
# evaluate_model is imported from src.evaluation.metrics
results = evaluate_model(
    model,
    test_custom,
    model_name='Baseline CNN',
    cmap='Blues',
    output_dir='../outputs'
)

print(f"\nFinal — Acc: {results['accuracy']:.4f} | "
      f"F1: {results['f1']:.4f} | AUC: {results['auc']:.4f}")